# UC Atlas: UMAP Visualization and Integration QC

Generates UMAP plots for the full UC atlas and lineage-specific subclusters
(Fig. 1B–D, Ext. Fig. 1), plus scIB `silhouette_batch` scoring comparing
PCA vs Harmony integration quality (Supplementary Table 3).

UMAP coordinates and embeddings are exported by `01_atlas_integration.R` and
the lineage subclustering scripts (02–04), then loaded here for plotting.

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import scib
import seaborn as sns
import matplotlib.pyplot as plt
from adjustText import adjust_text
from anndata import AnnData

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR = "/path/to/uc/scrna/output"
MYE_DIR  = f"{DATA_DIR}/myeloid"
FIB_DIR  = f"{DATA_DIR}/fibroblast"
T_DIR    = f"{DATA_DIR}/t_cells"
EPI_DIR  = f"{DATA_DIR}/epithelial"
QC_DIR   = f"{DATA_DIR}/qc"

## 1. Load UMAP coordinates and embeddings

In [ ]:
# Coarse-grain atlas UMAP (all cells, Harmony-integrated)
umap_all    = pd.read_csv(f"{DATA_DIR}/umap_atlas_coarse_anno.csv")
umap_uniteg = pd.read_csv(f"{DATA_DIR}/umap_atlas_coarse_anno_NOINTEG.csv")  # pre-integration baseline

# Lineage-specific UMAPs (from sub-clustering scripts)
umap_mye = pd.read_csv(f"{MYE_DIR}/umap_myeloid_iter3_anno.csv")
umap_fib = pd.read_csv(f"{FIB_DIR}/umap_fibroblast_iter4_anno.csv")
umap_t   = pd.read_csv(f"{T_DIR}/umap_t_iter2_anno.csv")
umap_epi = pd.read_csv(f"{EPI_DIR}/umap_epi_iter3_anno.csv")

In [ ]:
# PCA and Harmony embeddings exported by 06_integration_qc.R
pca     = pd.read_csv(f"{QC_DIR}/seurat_pca_embeddings.csv")
harmony = pd.read_csv(f"{QC_DIR}/seurat_harmony_embeddings.csv")

# Align metadata (only cells with embeddings)
meta = umap_all[["Unnamed: 0", "dataset", "cell_cat"]].rename(columns={"Unnamed: 0": "cell_id"})
meta = meta.merge(pca[["cell_id"]], on="cell_id", how="right")

em_df = pca.merge(harmony, on="cell_id").merge(meta, on="cell_id")

## 2. Color palettes

In [ ]:
# 15-color pastel palette (coarse cell types)
pastel_palette = [
    "#AEC6CF", "#FFB3C6", "#FF6666", "#FFCC99", "#99FF99",
    "#CCCCFF", "#b2e2eb", "#F0E68C", "#e4d4fa", "#FAD02E",
    "#D6AEDD", "#B3E5FC", "#FFDAB9", "#C1E1C1", "#F4C2C2"
]

# 35-color pastel palette (fine-grain labels)
pastel_palette_35 = [
    "#AEC6CF", "#FFB3C6", "#FF6666", "#FFCC99", "#99FF99",
    "#CCCCFF", "#FFFF99", "#F0E68C", "#E6E6FA", "#FAD02E",
    "#D6AEDD", "#B3E5FC", "#FFDAB9", "#C1E1C1", "#F4C2C2",
    "#F8B195", "#C5E384", "#B0E0E6", "#D9B3FF", "#FFE0AC",
    "#F7CAC9", "#C4FAF8", "#FFDEAD", "#B39EB5", "#FFB347",
    "#C3FDB8", "#FFDFDD", "#ADD8E6", "#E3E4FA", "#FFE4E1",
    "#F3D1DC", "#D2E3C8", "#EBD4CB", "#E0BBE4", "#FFFACD"
]

# Vibrant palette (lineage UMAPs)
my_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5"
]

## 3. Plot helper functions

In [ ]:
def plot_umap(data, x_col, y_col, hue_col, alpha=0.7, palette=pastel_palette,
              font=11, size=5):
    """UMAP scatter with centroid labels (no legend)."""
    plt.figure(figsize=(6, 6))
    scatter = sns.scatterplot(
        data=data, x=x_col, y=y_col, hue=hue_col,
        s=size, linewidth=0, palette=palette, alpha=alpha
    )
    handles, labels = scatter.get_legend_handles_labels()
    scatter.legend([], [], frameon=False)

    texts = []
    for label in labels:
        cluster_data = data[data[hue_col] == label]
        cx = cluster_data[x_col].mean()
        cy = cluster_data[y_col].mean()
        texts.append(plt.text(cx, cy, label, fontsize=font,
                               ha='center', va='center', color='black'))
    adjust_text(texts, arrowprops=dict(arrowstyle="->", color='black'))

    plt.grid(False)
    plt.xlabel('UMAP1')
    plt.ylabel('UMAP2')
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_umap_with_legend(data, x_col, y_col, hue_col, legend_x=0.02, legend_y=0.98,
                          alpha=0.7, title='', palette=pastel_palette, size=5):
    """UMAP scatter with an inside legend box (no centroid labels)."""
    plt.figure(figsize=(6, 6))
    scatter = sns.scatterplot(
        data=data, x=x_col, y=y_col, hue=hue_col,
        s=size, linewidth=0, palette=palette, alpha=alpha
    )
    handles, labels = scatter.get_legend_handles_labels()
    scatter.legend(
        handles=handles, labels=labels, title=title,
        loc='upper left', bbox_to_anchor=(legend_x, legend_y),
        markerscale=2, handlelength=1, frameon=True,
        edgecolor='white', facecolor='white', framealpha=0.6, fontsize='small'
    )
    plt.grid(False)
    plt.xlabel('UMAP1')
    plt.ylabel('UMAP2')
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_umap_label_and_legend(data, x_col, y_col, hue_col, alpha=0.7,
                               palette=None, font=11, title=''):
    """UMAP scatter with centroid labels AND an outside legend."""
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = sns.scatterplot(
        data=data, x=x_col, y=y_col, hue=hue_col,
        s=3, linewidth=0, palette=palette, alpha=alpha, ax=ax
    )
    handles, labels = scatter.get_legend_handles_labels()
    ax.legend(
        handles=handles, labels=labels, title=title,
        loc='center left', bbox_to_anchor=(1.05, 0.5),
        markerscale=2, handlelength=1, frameon=True,
        edgecolor='white', facecolor='white', framealpha=0.6,
        fontsize='small', ncol=2
    )
    texts = []
    for label in labels:
        cluster_data = data[data[hue_col] == label]
        cx = cluster_data[x_col].mean()
        cy = cluster_data[y_col].mean()
        texts.append(ax.text(cx, cy, label, fontsize=font, ha='center', va='center', color='black'))
    adjust_text(texts, ax=ax)
    ax.grid(False)
    ax.set_xlabel('UMAP1')
    ax.set_ylabel('UMAP2')
    ax.set_xticks([])
    ax.set_yticks([])
    plt.subplots_adjust(right=0.75)
    plt.show()

## 4. Atlas-level UMAP plots

In [ ]:
# Coarse cell category (Fig. 1B)
plot_umap(umap_all, 'UMAP_1', 'UMAP_2', 'cell_cat', size=3)

In [ ]:
# Dataset of origin – integrated vs unintegrated (Ext. Fig. 1)
plot_umap_with_legend(umap_all, 'UMAP_1', 'UMAP_2', 'dataset',
                      alpha=0.7, title='Dataset', size=3, palette=my_palette)

plot_umap_with_legend(umap_uniteg, 'umapnoharmony_1', 'umapnoharmony_2', 'dataset',
                      palette=my_palette, alpha=0.7, title='Dataset (no integration)',
                      legend_x=0.87, legend_y=0.37, size=3)

In [ ]:
# Original dataset labels (Ext. Fig. 1)
umap_all_nonull = umap_all[umap_all['label_orig_clean'] != 'Null']
plot_umap_label_and_legend(umap_all_nonull, 'UMAP_1', 'UMAP_2', 'label_orig_clean',
                           alpha=0.7, title='Original Label',
                           palette=pastel_palette_35, font=7)

## 5. Lineage-specific UMAP plots

In [ ]:
# Myeloid (Fig. 1C)
plot_umap(umap_mye, 'umapharmonymyeiter3_1', 'umapharmonymyeiter3_2', 'cell_type')

In [ ]:
# Fibroblast (Fig. 1D)
# Shorten long labels for display
label_map = {'NRG1+ Crypt Top Fibroblast': 'NRG1 CT Fib',
             'VSTM2A+ Crypt Top Fibroblast': 'VSTM2A CT Fib'}
umap_fib['cell_type_plot'] = umap_fib['cell_type'].replace(label_map)
plot_umap(umap_fib, 'umapharmonyconiter4_1', 'umapharmonyconiter4_2', 'cell_type_plot')

In [ ]:
# T cell
plot_umap(umap_t, 'umapharmonycd3titer2_1', 'umapharmonycd3titer2_2', 'cell_type')

In [ ]:
# Epithelial
plot_umap(umap_epi, 'umapharmonyepiiter2_1', 'umapharmonyepiiter2_2', 'cell_type')

## 6. scIB integration quality assessment (Supplementary Table 3)

Lineage-stratified `silhouette_batch` score measures how well batch effects are
removed while preserving cell-type identity within each lineage.
Higher score = better batch correction without over-mixing cell types.

In [ ]:
# Build AnnData from PCA + Harmony embeddings for scIB
pca_cols  = [c for c in em_df.columns if c.startswith("PC")]
harm_cols = [c for c in em_df.columns if c.startswith("harmony")]

adata = AnnData(X=np.zeros((em_df.shape[0], 1), dtype=np.float32))
adata.obs_names = em_df["cell_id"].astype(str).values
adata.obs["dataset"]  = em_df["dataset"].astype("category").values
adata.obs["cell_cat"] = em_df["cell_cat"].astype("category").values
adata.obsm["X_pca"]     = em_df[pca_cols].to_numpy(dtype=np.float32)
adata.obsm["X_harmony"] = em_df[harm_cols].to_numpy(dtype=np.float32)

In [ ]:
# Silhouette batch: PCA (pre-integration baseline)
sil_pca = scib.me.silhouette_batch(
    adata, batch_key="dataset", label_key="cell_cat", embed="X_pca"
)
print(f"Silhouette batch (PCA):     {sil_pca:.4f}")

In [ ]:
# Silhouette batch: Harmony (post-integration)
sil_har = scib.me.silhouette_batch(
    adata, batch_key="dataset", label_key="cell_cat", embed="X_harmony"
)
print(f"Silhouette batch (Harmony): {sil_har:.4f}")

In [ ]:
# Graph connectivity (Harmony)
sc.pp.neighbors(adata, use_rep="X_harmony")
gc_har = scib.me.graph_connectivity(adata, label_key="cell_cat")
print(f"Graph connectivity (Harmony): {gc_har:.4f}")